In [1]:
from google.colab import files

uploaded = files.upload()

for filename in uploaded.keys():
    print(f"Uploaded file: {filename}")

Saving input.txt to input.txt
Uploaded file: input.txt


In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F


'''HParams'''
batch_size = 64
block_size = 256
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
# head_size = 16
n_layer = 6
n_head = 6
dropout = 0.2


torch.manual_seed(1337)

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)

## Character level tokenization
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y

    data = train_data if split == 'train' else val_data

    ix = torch.randint(len(data) - block_size, (batch_size,))
    '''
    ix becomes a tensor with (highest value, (shape of integer))
    with len(data)=1130537, block-size=8, batch-size =32
    (1130537 - 8) = 1130529 and (32,)=(32,1)
    meaning 32 random integers between 0 to 1130529 in one tensor i.e, ix
    '''

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)

    '''
    loops over every integer in ix, and forms a new tensor array with that integer to integer+block size integers, and then all 32 are stacked together with x becoming the input and y with offset 1 as predicted val
    '''
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


class Head(nn.Module):
    '''One head of self attention'''

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T, C = x.shape
        k = self.key(x) #(B,T,C)
        q = self.query(x)

        # computing affinities (attention scores)

        wei = q @ k.transpose(-2,-1) * C**-0.5
        # BTC @ BCT -> BTT
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) #BTT
        wei = F.softmax(wei, dim =-1) #BTT
        wei = self.dropout(wei)

        #perform the weighted aggregation
        v = self.value(x) #BTC
        out = wei @ v #BTT @ BTC -> BTC

        return out


class MultiHeadAttention(nn.Module):
    """ Multiple heads of self attention in parallel"""
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim = -1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    """ a simple linear layer followed by a non-linearity"""

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),

        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication dimension, n_head : num of heads we'd like"""

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd// n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f  = nn.LayerNorm(n_embd)

        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):

        B,T  = idx.shape
        #idx(input indexes (xb)) and targets(yb) are both (Batch,Time) tensor of integers [taken when the model is called]
        token_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device = device)) #(T,C)

        x = token_emb + pos_emb #(B,T,C)
        x = self.blocks(x) #applying one head of self attention. (BTC)
        # x = self.ffwd(x) #BTC
        logits = self.lm_head(x) #(B,T,C = vocab size here)
        # logits are scores of next predicted token
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)

            loss = F.cross_entropy(logits, targets)
            #automatically compares the highest score vs targets

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):

            #crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets (whenever the remainder is 0)
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))

step 0: train loss 4.9491, val loss 4.9573
step 500: train loss 2.0946, val loss 2.0868
step 1000: train loss 1.5603, val loss 1.5887
step 1500: train loss 1.3671, val loss 1.4135
step 2000: train loss 1.2581, val loss 1.3297
step 2500: train loss 1.1817, val loss 1.2712
step 3000: train loss 1.1260, val loss 1.2419
step 3500: train loss 1.0794, val loss 1.2158
step 4000: train loss 1.0393, val loss 1.2069
step 4500: train loss 1.0015, val loss 1.1952

Sonia ran such an a goodishmelf-dampel, came out of relation. Here,
an intense, but he alarmed, he seemed to go without to, for the
face, and that float obstaken, how daste anything buttome builly stuffed
the struggle which he recovered himself or it alto the taff and at once
summer among times and others, no explainess--a very one’se fault,
since I can think of flunny. Marfly where did not leave him, that I
am playing to be for shome to be a student place of it? We will trether
if you comen to 


In [7]:
torch.save(model.state_dict(), "transformer_weights.pth")

In [ ]:
while True:
    import os
    if os.path.exists("transformer_weights.pth"):
        break

# from google.colab import files
files.download("transformer_weights.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>